## Part 1 - Inspect the dataset
Each row represents one penguin observation. species and island are categorical variables. bill_length_mm is bill length in mm, and body_mass_g is body mass in g. year is the numeric observation year.

In [1]:
options(width = 100)
source_file <- "penguins.csv"
source_hash <- unname(tools::md5sum(source_file))
penguins <- read.csv(source_file, na.strings = "NA", stringsAsFactors = FALSE)
original <- penguins
cat(R.version.string, "\n")
print(head(penguins, 6), row.names = FALSE)
cat("Dimensions:", dim(penguins), "\n")
print(vapply(penguins, function(x) paste(class(x), collapse = "/"), character(1)))
str(penguins)
print(colSums(is.na(penguins)))
stopifnot(identical(dim(penguins), c(344L, 8L)))
cat("PASS: 344 rows, 8 columns\n")

R version 4.6.1 (2026-06-24 ucrt) 
 species    island bill_length_mm bill_depth_mm flipper_length_mm body_mass_g    sex year
  Adelie Torgersen           39.1          18.7               181        3750   male 2007
  Adelie Torgersen           39.5          17.4               186        3800 female 2007
  Adelie Torgersen           40.3          18.0               195        3250 female 2007
  Adelie Torgersen             NA            NA                NA          NA   <NA> 2007
  Adelie Torgersen           36.7          19.3               193        3450 female 2007
  Adelie Torgersen           39.3          20.6               190        3650   male 2007
Dimensions: 344 8 
          species            island    bill_length_mm     bill_depth_mm flipper_length_mm 
      "character"       "character"         "numeric"         "numeric"         "integer" 
      body_mass_g               sex              year 
        "integer"       "character"         "integer" 
'data.frame':	344 obs. o

### Python/R comparison - inferred types
Python pandas reads flipper_length_mm and body_mass_g as float64 because ordinary NumPy integers cannot represent NaN. R reads these whole-number measurements as integer even with NA. year is int64 in Python and integer in R. Text columns are str in the pandas version used here and character in R; categorical meaning does not automatically imply a category/factor type. Decimal bill measurements are float64/double. Both implementations have identical dimensions and missing-value counts.

## Part 2 - Select rows and columns
Observations 5 through 10 refer to positions in the original file. Labels p01-p06 are illustrative and are not actual penguin identifiers.

In [2]:
cols <- c("species", "island", "flipper_length_mm")
selected <- penguins[, cols, drop = FALSE]
vector <- penguins[["flipper_length_mm"]]
one_column <- penguins[, "flipper_length_mm", drop = FALSE]
cat("Selection:", dim(selected), "\n")
cat("1D:", class(vector), "typeof:", typeof(vector), "length:", length(vector), "\n")
cat("Table:", class(one_column), "dim:", dim(one_column), "\n")
positions <- selected[5:10, , drop = FALSE]
print(positions, row.names = FALSE)
stopifnot(identical(dim(selected), c(344L, 3L)),
          identical(dim(positions), c(6L, 3L)), length(vector) == 344L,
          identical(dim(one_column), c(344L, 1L)))
small <- head(selected, 6)
rownames(small) <- sprintf("p%02d", 1:6)
stopifnot(!anyDuplicated(rownames(small)))
print(small[c("p02", "p05"), , drop = FALSE])
reverse <- small[6:1, , drop = FALSE]
print(reverse)
cat("First row by position:\n")
print(reverse[1, , drop = FALSE])
cat("p01 by label:\n")
print(reverse["p01", , drop = FALSE])
stopifnot(rownames(reverse)[1] == "p06",
          identical(reverse["p01", , drop = FALSE], small["p01", , drop = FALSE]))
cat("PASS: dimensions, unique labels and selections\n")

Selection: 344 3 
1D: integer typeof: integer length: 344 
Table: data.frame dim: 344 1 
 species    island flipper_length_mm
  Adelie Torgersen               193
  Adelie Torgersen               190
  Adelie Torgersen               181
  Adelie Torgersen               195
  Adelie Torgersen               193
  Adelie Torgersen               190
    species    island flipper_length_mm
p02  Adelie Torgersen               186
p05  Adelie Torgersen               193
    species    island flipper_length_mm
p06  Adelie Torgersen               190
p05  Adelie Torgersen               193
p04  Adelie Torgersen                NA
p03  Adelie Torgersen               195
p02  Adelie Torgersen               186
p01  Adelie Torgersen               181
First row by position:
    species    island flipper_length_mm
p06  Adelie Torgersen               190
p01 by label:
    species    island flipper_length_mm
p01  Adelie Torgersen               181
PASS: dimensions, unique labels and selections


After reversing the rows, p06 occupies the first position, while p01 still identifies the original first row. Positional selection follows order; label selection follows identity. Python positions start at 0 and the slice endpoint is excluded; R positions start at 1. In R, preserve a table with drop = FALSE or with penguins["flipper_length_mm"].

## Part 3 - Filter observations
Each filter starts independently from penguins. Missing measurements are explicitly excluded. For readability, the first ten retained rows are displayed; complete results are stored in f1, f2 and f3. Checks cover every retained row.

In [3]:
mask1 <- !is.na(penguins$flipper_length_mm) & (penguins$flipper_length_mm >= 210)
mask2 <- (penguins$species == "Gentoo") & !is.na(penguins$body_mass_g) &
         (penguins$body_mass_g >= 5000)
mask3 <- ((penguins$island == "Biscoe") | (penguins$island == "Dream")) &
         !is.na(penguins$bill_length_mm) & (penguins$bill_length_mm < 45)
f1 <- penguins[mask1, , drop = FALSE]
f2 <- penguins[mask2, , drop = FALSE]
f3 <- penguins[mask3, , drop = FALSE]
cat("F1 count:", nrow(f1), "\n")
print(head(f1[, c("species", "island", "flipper_length_mm")], 10), row.names = FALSE)
cat("F2 count:", nrow(f2), "\n")
print(head(f2[, c("species", "body_mass_g")], 10), row.names = FALSE)
cat("F3 count:", nrow(f3), "\n")
print(head(f3[, c("island", "bill_length_mm")], 10), row.names = FALSE)
stopifnot(all(!is.na(f1$flipper_length_mm)), all(f1$flipper_length_mm >= 210),
          all(f2$species == "Gentoo"), all(!is.na(f2$body_mass_g)),
          all(f2$body_mass_g >= 5000), all(f3$island %in% c("Biscoe", "Dream")),
          all(!is.na(f3$bill_length_mm)), all(f3$bill_length_mm < 45))
filter_counts <- c(nrow(f1), nrow(f2), nrow(f3))
cat("PASS: every retained row meets the conditions:", filter_counts, "\n")

F1 count: 114 
 species    island flipper_length_mm
  Adelie Torgersen               210
  Gentoo    Biscoe               211
  Gentoo    Biscoe               230
  Gentoo    Biscoe               210
  Gentoo    Biscoe               218
  Gentoo    Biscoe               215
  Gentoo    Biscoe               210
  Gentoo    Biscoe               211
  Gentoo    Biscoe               219
  Gentoo    Biscoe               215
F2 count: 67 
 species body_mass_g
  Gentoo        5700
  Gentoo        5700
  Gentoo        5400
  Gentoo        5200
  Gentoo        5150
  Gentoo        5550
  Gentoo        5850
  Gentoo        5850
  Gentoo        6300
  Gentoo        5350
F3 count: 127 
 island bill_length_mm
 Biscoe           37.8
 Biscoe           37.7
 Biscoe           35.9
 Biscoe           38.2
 Biscoe           38.8
 Biscoe           35.3
 Biscoe           40.6
 Biscoe           40.5
 Biscoe           37.9
 Biscoe           40.5
PASS: every retained row meets the conditions: 114 67 127 


### Logical mask containing NA
Direct indexing inserts an unknown row for each NA in the mask. subset() retains only TRUE and drops NA. The corrected mask explicitly excludes NA; na.rm = TRUE is not used as a substitute for checking an incorrect filter.

In [4]:
mass_mask <- penguins$body_mass_g >= 5000
direct <- penguins[mass_mask, , drop = FALSE]
via_subset <- subset(penguins, body_mass_g >= 5000)
correct_mask <- !is.na(mass_mask) & mass_mask
corrected <- penguins[correct_mask, , drop = FALSE]
unknown_rows <- rowSums(is.na(direct)) == ncol(direct)
cat("NA entries in the mask:", sum(is.na(mass_mask)), "\n")
cat("Direct indexing:", nrow(direct), "; all-NA rows:", sum(unknown_rows), "\n")
print(direct[unknown_rows, , drop = FALSE])
cat("subset:", nrow(via_subset), "; corrected mask:", nrow(corrected), "\n")
stopifnot(!anyNA(correct_mask), identical(corrected, via_subset),
          nrow(direct) == nrow(corrected) + sum(is.na(mass_mask)))
cat("PASS: the explicitly corrected mask matches subset()\n")

NA entries in the mask: 2 
Direct indexing: 69 ; all-NA rows: 2 
     species island bill_length_mm bill_depth_mm flipper_length_mm body_mass_g  sex year
NA      <NA>   <NA>             NA            NA                NA          NA <NA>   NA
NA.1    <NA>   <NA>             NA            NA                NA          NA <NA>   NA
subset: 67 ; corrected mask: 67 
PASS: the explicitly corrected mask matches subset()


In Python, comparing float NaN with a threshold returns False; in R, comparing NA returns NA. Both workflows therefore explicitly require a known measurement. With this handling, all three filter counts agree between Python and R; this agreement is also checked when building the reports.

## Part 4 - Build a working table
The conversion is vectorized: grams / 1000. Species is sorted ascending, mass descending, and missing masses last within each species. When both sorting keys tie, an explicit auxiliary position key preserves the original order. This auxiliary key is not retained in working.

In [5]:
working <- penguins[, c("species", "island", "flipper_length_mm", "body_mass_g")]
working$body_mass_kg <- working$body_mass_g / 1000
ord <- order(working$species, -working$body_mass_g, seq_len(nrow(working)),
             na.last = TRUE, method = "radix")
working <- working[ord, , drop = FALSE]
rownames(working) <- NULL
print(head(working, 10), row.names = FALSE)
known <- !is.na(working$body_mass_g)
stopifnot(all(abs(working$body_mass_kg[known] - working$body_mass_g[known]/1000) < 1e-12),
          nrow(working) == nrow(penguins),
          sum(is.na(working$body_mass_g)) == sum(is.na(penguins$body_mass_g)),
          ncol(penguins) == 8L, ncol(working) == 5L,
          identical(working$species, sort(working$species)), identical(penguins, original))
for (sp in unique(working$species)) {
    masses <- working$body_mass_g[working$species == sp]
    stopifnot(all(diff(masses[!is.na(masses)]) <= 0),
              all(diff(as.integer(is.na(masses))) >= 0))
}
cat("PASS: conversion (tol=1e-12), sorting, 344 rows, 5 columns,",
    sum(is.na(working$body_mass_g)), "missing masses; original unchanged\n")

 species    island flipper_length_mm body_mass_g body_mass_kg
  Adelie    Biscoe               197        4775        4.775
  Adelie    Biscoe               203        4725        4.725
  Adelie Torgersen               196        4700        4.700
  Adelie Torgersen               195        4675        4.675
  Adelie     Dream               184        4650        4.650
  Adelie     Dream               190        4600        4.600
  Adelie    Biscoe               191        4600        4.600
  Adelie Torgersen               197        4500        4.500
  Adelie     Dream               199        4475        4.475
  Adelie Torgersen               198        4450        4.450
PASS: conversion (tol=1e-12), sorting, 344 rows, 5 columns, 2 missing masses; original unchanged


## Part 5 - Export and verify
Exports omit row labels and write missing values as literal NA. We check dimensions, column names, types, missing values and all values, using a tolerance of 1e-12 for numerical comparisons. Separate filenames allow both languages to keep their exports in the same directory.

In [6]:
verify_restored <- function(restored) {
    cat("Dimensions:", dim(restored), "\n")
    print(vapply(restored, class, character(1)))
    print(colSums(is.na(restored)))
    stopifnot(identical(dim(restored), dim(working)),
              identical(names(restored), names(working)),
              identical(is.na(restored), is.na(working)))
    for (column in names(working)) {
        if (is.numeric(working[[column]])) {
            stopifnot(is.numeric(restored[[column]]))
            known <- !is.na(working[[column]])
            stopifnot(all(abs(working[[column]][known] - restored[[column]][known]) < 1e-12))
        } else {
            stopifnot(identical(working[[column]], restored[[column]]))
        }
    }
    stopifnot(identical(vapply(restored, class, character(1)),
                        vapply(working, class, character(1))))
    cat("PASS: dimensions, names, types, missing values and all values\n")
}
write.csv(working, "working_r.csv", row.names = FALSE, na = "NA")
restored <- read.csv("working_r.csv", na.strings = "NA", stringsAsFactors = FALSE)
verify_restored(restored)
write.table(working, "working_r_semicolon.csv", sep = ";", dec = ",",
            row.names = FALSE, col.names = TRUE, na = "NA")
restored_semicolon <- read.table("working_r_semicolon.csv", header = TRUE,
    sep = ";", dec = ",", na.strings = "NA", stringsAsFactors = FALSE)
verify_restored(restored_semicolon)

Dimensions: 344 5 
          species            island flipper_length_mm       body_mass_g      body_mass_kg 
      "character"       "character"         "integer"         "integer"         "numeric" 
          species            island flipper_length_mm       body_mass_g      body_mass_kg 
                0                 0                 2                 2                 2 
PASS: dimensions, names, types, missing values and all values
Dimensions: 344 5 
          species            island flipper_length_mm       body_mass_g      body_mass_kg 
      "character"       "character"         "integer"         "integer"         "numeric" 
          species            island flipper_length_mm       body_mass_g      body_mass_kg 
                0                 0                 2                 2                 2 
PASS: dimensions, names, types, missing values and all values


In [7]:
tryCatch({
    wrong <- read.csv("working_r_semicolon.csv")
    cat("Incorrect import - dimensions:", dim(wrong), "\n")
    print(names(wrong))
    print(head(wrong, 3))
    stopifnot(!identical(dim(wrong), dim(working)) || !identical(names(wrong), names(working)))
}, error = function(e) cat("Expected import error:", conditionMessage(e), "\n"))
# Corrected import with the matching separator and decimal comma.
corrected_import <- read.table("working_r_semicolon.csv", header = TRUE,
    sep = ";", dec = ",", na.strings = "NA", stringsAsFactors = FALSE)
verify_restored(corrected_import)
stopifnot(identical(unname(tools::md5sum(source_file)), source_hash),
          identical(penguins, original))
cat("PASS: the source CSV and penguins object are unchanged\n")
cat("Filter counts:", filter_counts, "\n")

Expected import error: duplicate 'row.names' are not allowed 
Dimensions: 344 5 
          species            island flipper_length_mm       body_mass_g      body_mass_kg 
      "character"       "character"         "integer"         "integer"         "numeric" 
          species            island flipper_length_mm       body_mass_g      body_mass_kg 
                0                 0                 2                 2                 2 
PASS: dimensions, names, types, missing values and all values
PASS: the source CSV and penguins object are unchanged
Filter counts: 114 67 127 


### CSV interpretation and submission preparation
The default comma-separated parser does not recognize semicolons and mistakes decimal commas for separators. This produces an incorrect table or a parser error; expected errors are handled so execution can continue. The correction uses sep=";" and decimal="," in Python, or dec="," in R.
CSV preserves textual representations of values, column names and row/column order. It does not preserve data types, units, categories/factors, index/row labels or notebook metadata. In these exports, missing values are recognized using the agreed NA string. Other datasets may need to distinguish literal text "NA" from a missing value. Types are inferred again and must be verified or specified explicitly.
Python/R comparison: both versions have 344 x 8 in the source and 344 x 5 in working, identical filters, preserved missing values and equal values after both exports. Python uses Series/DataFrame; R uses vector/data.frame and drop=FALSE. R additionally demonstrates how indexing with an NA mask differs from subset().
The notebooks and PDFs were generated from code executed in fresh local processes. To complete the required Colab step, open each notebook, select the Python 3 / R runtime, upload penguins.csv, restart the session and run all cells. Then export the report to PDF again if needed. The ZIP contains exactly two PDF reports; CSV files remain in the working directory.